In [ ]:
import os
import pandas as pd
#import pycountry

In [ ]:
# set the directory where the CSV files are located
path = '/content/drive/MyDrive/INFLUENZA RAG/USA'

# get all CSV files in the directory
files = [file for file in os.listdir(path) if file.endswith('.xls')]

# create an empty list to store the dataframes
dfs = []

# loop through the CSV files and read them into pandas dataframes
for file in files:
    df = pd.read_excel(os.path.join(path, file))
    dfs.append(df)

# concatenate all the dataframes into a single dataframe
df_all = pd.concat(dfs, ignore_index=True)

df_all.to_csv('/content/drive/MyDrive/INFLUENZA RAG/USA/combined.csv', index=False)


In [ ]:
df_all.head()

,Isolate_Id,PB2 Segment_Id,PB1 Segment_Id,PA Segment_Id,HA Segment_Id,NP Segment_Id,NA Segment_Id,MP Segment_Id,NS Segment_Id,HE Segment_Id,...,Patient_Status,Zip_Code,Outbreak,Pathogen_Test_Info,Is_Vaccinated,Human_Specimen_Source,Animal_Specimen_Source,Animal_Health_Status,Domestic_Status,PMID
0,EPI_ISL_19564132,EPI3659704|A/Washington/S26109/2022_PB2,EPI3659702|A/Washington/S26109/2022_PB1,EPI3659699|A/Washington/S26109/2022_PA,EPI3659700|A/Washington/S26109/2022_HA,EPI3659701|A/Washington/S26109/2022_NP,EPI3659698|A/Washington/S26109/2022_NA,EPI3659705|A/Washington/S26109/2022_MP,EPI3659703|A/Washington/S26109/2022_NS,NaN,...,NaN,NaN,NaN,NaN,NaN,Nasal swab,NaN,NaN,NaN,NaN
1,EPI_ISL_19564131,EPI3659696|A/Washington/S26095/2022_PB2,EPI3659694|A/Washington/S26095/2022_PB1,EPI3659691|A/Washington/S26095/2022_PA,EPI3659692|A/Washington/S26095/2022_HA,EPI3659693|A/Washington/S26095/2022_NP,EPI3659690|A/Washington/S26095/2022_NA,EPI3659697|A/Washington/S26095/2022_MP,EPI3659695|A/Washington/S26095/2022_NS,NaN,...,NaN,NaN,NaN,NaN,NaN,Nasal swab,NaN,NaN,NaN,NaN
2,EPI_ISL_19564130,EPI3659688|A/Washington/S26094/2022_PB2,EPI3659686|A/Washington/S26094/2022_PB1,EPI3659683|A/Washington/S26094/2022_PA,EPI3659684|A/Washington/S26094/2022_HA,EPI3659685|A/Washington/S26094/2022_NP,EPI3659682|A/Washington/S26094/2022_NA,EPI3659689|A/Washington/S26094/2022_MP,EPI3659687|A/Washington/S26094/2022_NS,NaN,...,NaN,NaN,NaN,NaN,NaN,Nasal swab,NaN,NaN,NaN,NaN
3,EPI_ISL_19564129,EPI3659680|A/Washington/S26093/2022_PB2,EPI3659678|A/Washington/S26093/2022_PB1,EPI3659675|A/Washington/S26093/2022_PA,EPI3659676|A/Washington/S26093/2022_HA,EPI3659677|A/Washington/S26093/2022_NP,EPI3659674|A/Washington/S26093/2022_NA,EPI3659681|A/Washington/S26093/2022_MP,EPI3659679|A/Washington/S26093/2022_NS,NaN,...,NaN,NaN,NaN,NaN,NaN,Nasal swab,NaN,NaN,NaN,NaN
4,EPI_ISL_19564128,EPI3659672|A/Washington/S26092/2022_PB2,EPI3659670|A/Washington/S26092/2022_PB1,EPI3659667|A/Washington/S26092/2022_PA,EPI3659668|A/Washington/S26092/2022_HA,EPI3659669|A/Washington/S26092/2022_NP,EPI3659666|A/Washington/S26092/2022_NA,EPI3659673|A/Washington/S26092/2022_MP,EPI3659671|A/Washington/S26092/2022_NS,NaN,...,NaN,NaN,NaN,NaN,NaN,Nasal swab,NaN,NaN,NaN,NaN


In [ ]:
df_all.shape

(82263, 56)

In [ ]:
columns_of_interest=['Subtype','Location','Collection_Date']

In [ ]:
df_all=df_all[columns_of_interest]

In [ ]:
df_all.columns

Index(['Subtype', 'Location', 'Collection_Date'], dtype='object')

In [ ]:
df = df_all[["Subtype", "Location", "Collection_Date"]].copy()

In [ ]:
!pip install langchain langchain-community langchain-anthropic faiss-cpu sentence-transformers langchain-text-splitters -q

In [ ]:
import os
import pandas as pd

from langchain_community.embeddings   import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_anthropic              import ChatAnthropic
from langchain_core.prompts           import PromptTemplate
from langchain_core.documents         import Document
from langchain_core.output_parsers    import StrOutputParser
from langchain_core.runnables         import RunnablePassthrough
from langchain_text_splitters         import RecursiveCharacterTextSplitter  # ← fixed

In [ ]:
import os
import pandas as pd

from langchain_community.embeddings   import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_anthropic              import ChatAnthropic
from langchain_core.prompts           import PromptTemplate
from langchain_core.documents         import Document
from langchain_core.output_parsers    import StrOutputParser
from langchain_core.runnables         import RunnablePassthrough
from langchain_text_splitters         import RecursiveCharacterTextSplitter

# ── API Key ──────────────────────────────────────────────────
os.environ["ANTHROPIC_API_KEY"] = ""

# ── Clean df_all ─────────────────────────────────────────────
df = df_all[["Subtype", "Location", "Collection_Date"]].copy()
df["Subtype"]         = df["Subtype"].astype(str).str.strip().str.upper()
df["Location"]        = df["Location"].astype(str).str.strip().str.title()
df["Collection_Date"] = pd.to_datetime(df["Collection_Date"], errors="coerce")
df                    = df.dropna(subset=["Collection_Date"])
df["Year"]            = df["Collection_Date"].dt.year
df["Month"]           = df["Collection_Date"].dt.month
df["Season"]          = df["Month"].map({
    12:"Winter", 1:"Winter",  2:"Winter",
    3: "Spring", 4:"Spring",  5:"Spring",
    6: "Summer", 7:"Summer",  8:"Summer",
    9: "Autumn", 10:"Autumn", 11:"Autumn"
})
print(f"✅ {len(df)} total records loaded | Subtypes: {sorted(df['Subtype'].unique())}")

# ── SAMPLE FOR FAST TESTING ───────────────────────────────────
# Remove this line when you're ready to run on the full dataset
df = df.sample(78284, random_state=42).reset_index(drop=True)
print(f"✅ Using sample of {len(df)} records for testing")

# ── Pattern Summary ───────────────────────────────────────────
subtype_counts                = df["Subtype"].value_counts().to_dict()
top_locations                 = df["Location"].value_counts().head(10).to_dict()
yearly_trend                  = df.groupby("Year")["Subtype"].count().to_dict()
seasonal_pattern              = df.groupby("Season")["Subtype"].count().to_dict()
dominant_subtype_per_season   = df.groupby("Season")["Subtype"].agg(lambda x: x.value_counts().idxmax()).to_dict()
dominant_subtype_per_location = df.groupby("Location")["Subtype"].agg(lambda x: x.value_counts().idxmax()).to_dict()

PATTERN_SUMMARY = f"""
=== FULL-DATASET PATTERN SUMMARY ({len(df)} records) ===
[ Subtype Frequencies ]
{chr(10).join(f"  {k}: {v} cases" for k,v in sorted(subtype_counts.items(), key=lambda x:-x[1]))}
[ Top 10 Locations ]
{chr(10).join(f"  {k}: {v} cases" for k,v in top_locations.items())}
[ Yearly Trend ]
{chr(10).join(f"  {k}: {v} cases" for k,v in sorted(yearly_trend.items()))}
[ Seasonal Distribution ]
{chr(10).join(f"  {k}: {v} cases" for k,v in seasonal_pattern.items())}
[ Dominant Subtype per Season ]
{chr(10).join(f"  {k}: {v}" for k,v in dominant_subtype_per_season.items())}
[ Dominant Subtype per Location (top 15) ]
{chr(10).join(f"  {k}: {v}" for k,v in list(dominant_subtype_per_location.items())[:15])}
""".strip()
print(PATTERN_SUMMARY)

# ── Build FAISS Vector Store ──────────────────────────────────
def row_to_text(row):
    return (f"Influenza subtype {row['Subtype']} collected in {row['Location']} "
            f"on {row['Collection_Date'].strftime('%B %d, %Y')} "
            f"({row['Season']} {int(row['Year'])}).")

documents = [
    Document(
        page_content=row_to_text(row),
        metadata={
            "subtype":  row["Subtype"],
            "location": row["Location"],
            "date":     str(row["Collection_Date"].date()),
            "season":   row["Season"],
            "year":     int(row["Year"]),
        }
    )
    for _, row in df.iterrows()
]

chunks       = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(documents)
embeddings   = HuggingFaceEmbeddings(
                   model_name="sentence-transformers/all-MiniLM-L6-v2",
                   model_kwargs={"device": "cpu"},
                   encode_kwargs={"batch_size": 64}
               )
vector_store = FAISS.from_documents(chunks, embeddings)
retriever    = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})
print(f"✅ FAISS index built from {len(chunks)} chunks")

# ── Claude LLM ────────────────────────────────────────────────
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0, max_tokens=2048)

# ── Prompt ────────────────────────────────────────────────────
PROMPT_TEMPLATE = f"""You are an influenza surveillance analyst.
You have two information sources:

1. PATTERN SUMMARY (entire dataset):
{PATTERN_SUMMARY}

2. RETRIEVED RECORDS (most relevant to the question):
{{context}}

Use the Pattern Summary for trend/frequency/seasonal questions.
Use Retrieved Records for specific lookups.

Question: {{question}}
Answer:"""

prompt = PromptTemplate(template=PROMPT_TEMPLATE, input_variables=["context", "question"])

# ── LCEL Chain ────────────────────────────────────────────────
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Pipeline ready!\n")

# ── ask() helper ─────────────────────────────────────────────
def ask(question: str):
    print(f"\n❓ {question}")
    print("─" * 65)
    answer = rag_chain.invoke(question)
    print(f"💬 {answer}")
    print("─" * 65)

# ── Example queries ───────────────────────────────────────────
ask("What is the most common influenza subtype?")
ask("Are there any seasonal patterns in the data?")
ask("Which locations have the most cases?")
ask("Has any subtype been increasing or decreasing over time?")

In [ ]:
# ask("Compare H1N1 vs B trends from October 17, 2022 to December 26, 2022")

## Experimenting with Different Models and Parameters

In [ ]:
import os
import pandas as pd

from langchain_community.embeddings   import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_anthropic              import ChatAnthropic
from langchain_core.prompts           import PromptTemplate
from langchain_core.documents         import Document
from langchain_core.output_parsers    import StrOutputParser
from langchain_core.runnables         import RunnablePassthrough
from langchain_text_splitters         import RecursiveCharacterTextSplitter

# ── API Key ──────────────────────────────────────────────────
os.environ["ANTHROPIC_API_KEY"] = ""

# ── Clean df_all ─────────────────────────────────────────────
df = df_all[["Subtype", "Location", "Collection_Date"]].copy()
df["Subtype"]         = df["Subtype"].astype(str).str.strip().str.upper()
df["Location"]        = df["Location"].astype(str).str.strip().str.title()
df["Collection_Date"] = pd.to_datetime(df["Collection_Date"], errors="coerce")
df                    = df.dropna(subset=["Collection_Date"])
df["Year"]            = df["Collection_Date"].dt.year
df["Month"]           = df["Collection_Date"].dt.month
df["Season"]          = df["Month"].map({
    12:"Winter", 1:"Winter",  2:"Winter",
    3: "Spring", 4:"Spring",  5:"Spring",
    6: "Summer", 7:"Summer",  8:"Summer",
    9: "Autumn", 10:"Autumn", 11:"Autumn"
})
print(f"✅ {len(df)} total records loaded | Subtypes: {sorted(df['Subtype'].unique())}")

# ── SAMPLE FOR FAST TESTING ───────────────────────────────────
df = df.sample(78284, random_state=42).reset_index(drop=True)
print(f"✅ Using sample of {len(df)} records for testing")

# ── Pattern Summary ───────────────────────────────────────────
subtype_counts                = df["Subtype"].value_counts().to_dict()
top_locations                 = df["Location"].value_counts().head(10).to_dict()
yearly_trend                  = df.groupby("Year")["Subtype"].count().to_dict()
seasonal_pattern              = df.groupby("Season")["Subtype"].count().to_dict()
dominant_subtype_per_season   = df.groupby("Season")["Subtype"].agg(lambda x: x.value_counts().idxmax()).to_dict()
dominant_subtype_per_location = df.groupby("Location")["Subtype"].agg(lambda x: x.value_counts().idxmax()).to_dict()

PATTERN_SUMMARY = f"""
=== FULL-DATASET PATTERN SUMMARY ({len(df)} records) ===
[ Subtype Frequencies ]
{chr(10).join(f"  {k}: {v} cases" for k,v in sorted(subtype_counts.items(), key=lambda x:-x[1]))}
[ Top 10 Locations ]
{chr(10).join(f"  {k}: {v} cases" for k,v in top_locations.items())}
[ Yearly Trend ]
{chr(10).join(f"  {k}: {v} cases" for k,v in sorted(yearly_trend.items()))}
[ Seasonal Distribution ]
{chr(10).join(f"  {k}: {v} cases" for k,v in seasonal_pattern.items())}
[ Dominant Subtype per Season ]
{chr(10).join(f"  {k}: {v}" for k,v in dominant_subtype_per_season.items())}
[ Dominant Subtype per Location (top 15) ]
{chr(10).join(f"  {k}: {v}" for k,v in list(dominant_subtype_per_location.items())[:15])}
""".strip()
print(PATTERN_SUMMARY)

# ── Build FAISS Vector Store ──────────────────────────────────
def row_to_text(row):
    return (f"Influenza subtype {row['Subtype']} collected in {row['Location']} "
            f"on {row['Collection_Date'].strftime('%B %d, %Y')} "
            f"({row['Season']} {int(row['Year'])}).")

documents = [
    Document(
        page_content=row_to_text(row),
        metadata={
            "subtype":  row["Subtype"],
            "location": row["Location"],
            "date":     str(row["Collection_Date"].date()),
            "season":   row["Season"],
            "year":     int(row["Year"]),
        }
    )
    for _, row in df.iterrows()
]

chunks     = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(documents)
embeddings = HuggingFaceEmbeddings(
                 model_name="sentence-transformers/all-mpnet-base-v2",
                 model_kwargs={"device": "cuda"},
                 encode_kwargs={"batch_size": 256}
             )

vector_store = FAISS.from_documents(chunks, embeddings)
retriever    = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
print(f"✅ FAISS index built from {len(chunks)} chunks")

# ══════════════════════════════════════════════════════════════
# ██  FIXED: Use current model ID (April 2026)               ██
# ══════════════════════════════════════════════════════════════
# The ONLY valid model IDs as of April 2026 are:
#   claude-haiku-4-5-20251001   ← cheapest, best for $5 budget
#   claude-sonnet-4-6           ← mid-tier
#   claude-opus-4-6             ← most expensive
# ══════════════════════════════════════════════════════════════

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",   # ← THIS IS THE CORRECT MODEL ID
    temperature=0.2,
    max_tokens=2048
)
print("✅ LLM configured: claude-haiku-4-5-20251001")

# ── Prompt ────────────────────────────────────────────────────
PROMPT_TEMPLATE = f"""You are an influenza surveillance analyst.
You have two information sources:

1. PATTERN SUMMARY (entire dataset):
{PATTERN_SUMMARY}

2. RETRIEVED RECORDS (most relevant to the question):
{{context}}

Use the Pattern Summary for trend/frequency/seasonal questions.
Use Retrieved Records for specific lookups.

Question: {{question}}
Answer:"""

prompt = PromptTemplate(template=PROMPT_TEMPLATE, input_variables=["context", "question"])

# ── LCEL Chain ────────────────────────────────────────────────
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Pipeline ready!\n")

# ── ask() helper ─────────────────────────────────────────────
def ask(question: str):
    print(f"\n❓ {question}")
    print("─" * 65)
    answer = rag_chain.invoke(question)
    print(f"💬 {answer}")
    print("─" * 65)

# ── Example queries ───────────────────────────────────────────
ask("What is the most common influenza subtype?")
ask("Are there any seasonal patterns in the data?")
ask("Which locations have the most cases?")
ask("Has any subtype been increasing or decreasing over time?")

✅ 78284 total records loaded | Subtypes: ['A', 'A / H1', 'A / H1N1', 'A / H1N2', 'A / H3', 'A / H3N1', 'A / H3N2', 'A / H3N3', 'A / H3N4', 'A / H3N5', 'A / H5', 'A / H5N1', 'B', 'C']
✅ Using sample of 78284 records for testing
=== FULL-DATASET PATTERN SUMMARY (78284 records) ===
[ Subtype Frequencies ]
  A / H3N2: 36979 cases
  A / H1N1: 27633 cases
  B: 13425 cases
  A / H3: 104 cases
  A / H5N1: 62 cases
  A / H1: 52 cases
  A / H1N2: 17 cases
  A: 4 cases
  A / H3N1: 2 cases
  A / H5: 2 cases
  A / H3N5: 1 cases
  A / H3N3: 1 cases
  A / H3N4: 1 cases
  C: 1 cases
[ Top 10 Locations ]
  North America / United States / Michigan: 9238 cases
  North America / United States: 7069 cases
  North America / United States / California: 4045 cases
  North America / United States / Texas: 3984 cases
  North America / United States / Washington: 2998 cases
  North America / United States / Pennsylvania: 2907 cases
  North America / United States / Wisconsin: 2807 cases
  North America / United 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index built from 78284 chunks
✅ LLM configured: claude-haiku-4-5-20251001
✅ Pipeline ready!


❓ What is the most common influenza subtype?
─────────────────────────────────────────────────────────────────
💬 # Answer

Based on the **PATTERN SUMMARY** of the entire dataset (78,284 records):

## **A / H3N2 is the most common influenza subtype**

**Subtype Breakdown:**
- **A / H3N2: 36,979 cases (47.2%)** ← Most dominant
- A / H1N1: 27,633 cases (35.3%)
- B: 13,425 cases (17.1%)
- All other subtypes: 247 cases combined (<0.4%)

### Key Observations:
- **A / H3N2 accounts for nearly half of all influenza cases** in the dataset
- It is the dominant subtype across **all seasons** (Autumn, Spring, Summer, and Winter)
- It is the dominant subtype across **all major geographic locations** in North America and the United States
- The second most common subtype (A / H1N1) has approximately **25% fewer cases** than A / H3N2

This makes A / H3N2 the clear epidemiological leader in this surve

In [ ]:
ask("Compare H1N1 vs B trends from October 17, 2022 to December 26, 2022")


❓ Compare H1N1 vs B trends from October 17, 2022 to December 26, 2022
─────────────────────────────────────────────────────────────────
💬 # H1N1 vs B Trends Analysis (October 17, 2022 - December 26, 2022)

## Data Limitations
The retrieved records provided do not contain specific cases from the October 17 - December 26, 2022 timeframe, so I cannot provide a detailed comparison for this exact period.

## Analysis Based on Pattern Summary

### Overall Frequency (Full Dataset)
- **A / H1N1**: 27,633 cases (35.3% of all influenza)
- **B**: 13,425 cases (17.1% of all influenza)
- **Ratio**: H1N1 is approximately **2:1** more prevalent than B

### Seasonal Context (October-December = Winter)
The queried period falls within **Winter**, which shows:
- **Winter total**: 43,397 cases
- **Dominant subtype**: A / H3N2 (not H1N1 or B)

### 2022 Year-Round Data
- **2022 total**: 13,577 cases
- H1N1 and B frequencies for 2022 specifically are not disaggregated in the summary

## Key Observations
1. 